# Suite DRV — The layout driver

`Driver` follows a diagram's journal, one entry at a time: a transaction's effect, which it sections into the logical layout's changes, anticipating what they remove, then applying it and calling the container layout and placement functions they imply, knowing nothing of the step that wrote it. Every bench suite runs through it; these cases are the entries no single step writes. mbse-basis [CAP-261: layout driven by any journal entry](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-261).

In [ ]:
from support import Bench, grid_symbol, symbol, text
from mbse.Diagrams.Framework import Container, Logical

b = Bench()
L = b.source.Logical

<a id="DRV-01"></a>
## DRV-01 · One entry that makes a top-level shape, a child of it, and sets the diagram's defaults: its grids configured first, then the defaults, then each shape placed, in the order the entry names them

In [ ]:
d = symbol("")
assert b.logical(["p", "p.q", "p.r"], lambda _: getattr(_, d).grid_span(1).grid_grows("columns")) == '[{"kind":"grid","container":"overview","span":1,"grows":"columns","cell_rotation":0.0},{"kind":"grid","container":"p","span":1,"grows":"columns","cell_rotation":0.0},{"kind":"place","shape":"p","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1},{"kind":"place","shape":"q","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1},{"kind":"place","shape":"r","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert text([b.cells(""), b.cells("p")]) == '["[[[0,0],[[\\"p\\",0]],null,1,1]]","[[[0,0],[[\\"q\\",0]],null,1,1],[[0,1],[[\\"r\\",1]],null,1,1]]"]'
assert text(b.replay()) == '[]'

<a id="DRV-02"></a>
## DRV-02 · A spec turned in place is unjoined before the entry and adjoined after it, by its new rotation

In [ ]:
assert b.spec("p", "q", "r", 90.0) == '[{"kind":"rule","from_shape":"q","to_shape":"r","degrees":90.0,"status":"held"}]'
spec = {"sp": L.OfAdjacency}
turned = b._step("logical", ("turn the spec",), ["p.q"],
                 lambda _: getattr(_, symbol("p.q")).owns({"to r 90": lambda e: e.owned(_.sp)}),
                 lambda _: _.sp.rotation(lambda r: r.degrees(270.0)), spec=spec)
assert turned == '[{"kind":"rule","from_shape":"q","to_shape":"r","degrees":90.0,"status":"dropped"},{"kind":"move","shape":"r","from":[0,1],"to":[0,0]},{"kind":"move","shape":"q","from":[0,0],"to":[0,1]},{"kind":"rule","from_shape":"q","to_shape":"r","degrees":270.0,"status":"met"}]'
assert text([b.cells("p"), b.rules("p")]) == '["[[[0,0],[[\\"r\\",1]],null,1,1],[[0,1],[[\\"q\\",0]],null,1,1]]","[[\\"q\\",\\"r\\",270.0,false]]"]'
assert text(b.replay()) == '[]'

<a id="DRV-03"></a>
## DRV-03 · What it cannot follow it leaves: a member of a diagram with no grid is not placed, nor a spec between shapes with no cells, and nothing is refused

In [ ]:
lone = Bench()
s = symbol("")
assert lone._step("logical", ("a member, no grid",), ["x", "y"],
                  lambda _: getattr(_, s).shapes([getattr(_, symbol("x")), getattr(_, symbol("y"))])) == '[]'
assert lone._step("logical", ("a spec, no cells",), ["x", "y"],
                  lambda _: getattr(_, symbol("x")).owns({"to y 90": lambda e: e.owned(_.sp)}),
                  lambda _: _.sp.from_shape([getattr(_, symbol("x"))]).to_shape([getattr(_, symbol("y"))])
                  .rotation(lambda r: r.degrees(90.0)), spec={"sp": lone.source.Logical.OfAdjacency}) == '[]'
assert text([Logical.grid_of(lone.overview), Container.cell_of(lone.shape("x")), len(list(lone.shape("x").adjacent_to))]) == '[null,null,1]'
assert text(lone.replay()) == '[]'